# Module 4 — Conversation State and Safe Memory

This notebook uses fictional support messages and scripted model output. From `module-04/`, install `python -m pip install -e ".[dev]"`. No API key is needed. Work through Build → Break → Measure → Improve.


## 1. Start a session

The owner ID is supplied by this local demo. A real application must get it from authentication. The store is process memory only and expires after 30 minutes by default.


In [ ]:
import json
from state_lab.router import SupportRouter
from state_lab.service import SupportService
from state_lab.store import MemoryStore, SessionNotFound

class ScriptedGenerator:
    def __init__(self, outputs):
        self.outputs = iter(outputs)
        self.inputs = []
    def generate(self, *, instructions, user_input):
        self.inputs.append(json.loads(user_input))
        return json.dumps(next(self.outputs))

def proposal(route, updates=None, question=""):
    return {
        "intent": "order_status", "route": route,
        "missing_information": ["order_reference"] if question else [],
        "pending_question": question, "response_draft": "",
        "needs_human_review": route == "handoff", "fact_updates": updates or [],
    }

script = ScriptedGenerator([
    proposal("ask_clarifying_question", question="What is the order number?"),
    proposal("handoff", [{"key":"order_reference","value":"A123","operation":"set"}]),
    proposal("handoff", [{"key":"order_reference","value":"B456","operation":"correct"}]),
])
service = SupportService(MemoryStore(), SupportRouter(script))
owner = "fictional-alice"
session = service.start(owner_id=owner)["session_id"]
print(service.review(owner_id=owner, session_id=session))


## 2. Build: ask, resume, correct

The first turn records a pending question. The second records a *reported* order number. The third explicitly supersedes it. Notice that the handoff response is application-controlled and never claims an action occurred.


In [ ]:
for message in ("Where is my order?", "Order A123", "Correction: Order B456"):
    result = service.turn(owner_id=owner, session_id=session, message=message)
    print(result.route, result.response)
print(json.dumps(service.review(owner_id=owner, session_id=session), indent=2))
assert service.review(owner_id=owner, session_id=session)["current_facts"]["order_reference"]["value"] == "B456"


## 3. Inspect the bounded context

The router sends current reported facts and the pending question, rather than the raw transcript. A model output remains a proposal, not verified order data.


In [ ]:
for i, item in enumerate(script.inputs, 1):
    print(i, json.dumps(item["state"], ensure_ascii=False))


## 4. Break: another owner tries this session

A session ID alone must not grant access. The store scopes reads and writes by the caller's authenticated owner ID.


In [ ]:
try:
    service.review(owner_id="fictional-bob", session_id=session)
except SessionNotFound:
    print("Other owner cannot read this session")
else:
    raise AssertionError("Cross-owner access succeeded")


## 5. Review, correct, delete

The explicit correction API does not need another model call. Delete verifies removal only from this in-memory store.


In [ ]:
print(service.correct(owner_id=owner, session_id=session,
                      key="order_reference", value="C789")["current_facts"])
assert service.delete(owner_id=owner, session_id=session)
try:
    service.review(owner_id=owner, session_id=session)
except SessionNotFound:
    print("This process no longer has the session")


## 6. Measure and improve

Run `python -m pytest -q` from `module-04/`. The test suite includes 26 scripted synthetic scenarios and focused isolation, expiration, and invalid-output tests. Write down one failure that a scripted model cannot reveal, then add a new fictional case or propose an independent verification source. Optional live evaluation is documented in `README.md` and may incur API charges.
